# Lab 03 — Search by Meaning, Not Just Words

A pretrained model may create embeddings. Your job is to implement cosine similarity and deterministic top-*k* retrieval in NumPy.

**Rules:** CPU only; no vector database; no hosted embedding API; no library nearest-neighbour/search implementation.

In [ ]:
%pip -q install sentence-transformers numpy

In [ ]:
from time import perf_counter
import numpy as np
from sentence_transformers import SentenceTransformer

np.set_printoptions(precision=4, suppress=True)
RNG = np.random.default_rng(42)
MODEL_NAME = "all-MiniLM-L6-v2"

## Challenge corpus

The data is deterministic and deliberately contains paraphrases. It is lab-only data, not the InstaKill domain model.

In [ ]:
TOPICS = {
    "food": [
        "A budget-friendly restaurant opened near the station.",
        "Looking for an affordable place to eat dinner tonight.",
        "The cheapest lunch menu I found this week costs under ten euros.",
    ],
    "running": [
        "Easy jogging route beside the river for beginners.",
        "My first 5K training plan starts with short runs and walks.",
        "Anyone know a calm place for an evening run?",
    ],
    "python": [
        "I finally understood why a Python dictionary needs hashable keys.",
        "Small NumPy experiment: vectorized code was much faster than my loop.",
        "Looking for a clear explanation of Python type hints.",
    ],
    "music": [
        "Quiet jazz playlist for a long study session.",
        "Any recommendations for live music this weekend?",
        "This new album is perfect background sound while reading.",
    ],
    "travel": [
        "Best low-cost train trip for a one-day escape?",
        "I need ideas for a weekend hike without a car.",
        "What should I pack for a short mountain trip in autumn?",
    ],
    "books": [
        "Recommend a short science-fiction novel with a strong ending.",
        "I want a book that explains machine learning without too much math.",
        "Which library has a good quiet corner for reading?",
    ],
}

def make_demo_posts(repetitions=200):
    posts = []
    post_id = 1
    for topic, templates in TOPICS.items():
        for repeat in range(repetitions):
            template = templates[repeat % len(templates)]
            posts.append({
                "post_id": post_id,
                "user_id": (repeat % 37) + 1,
                "topic": topic,
                "content": f"{template} [community note {repeat + 1}]",
            })
            post_id += 1
    RNG.shuffle(posts)
    return posts

posts = make_demo_posts()
texts = [post["content"] for post in posts]
assert len(posts) >= 1_000
print(f"{len(posts)} posts; example: {posts[0]}")

1200 posts; example: {'post_id': 70, 'user_id': 33, 'topic': 'food', 'content': 'A budget-friendly restaurant opened near the station. [community note 70]'}


## Baseline: keyword overlap

This baseline is intentionally simple. You may inspect it, but do not use it as your semantic solution.

In [ ]:
def keyword_search(query, posts, k=5):
    query_terms = set(query.lower().split())
    scored = []
    for post in posts:
        score = len(query_terms & set(post["content"].lower().split()))
        scored.append((score, post["post_id"], post))
    scored.sort(key=lambda item: (-item[0], item[1]))
    return [post for score, _, post in scored[:k] if score > 0]

keyword_search("cheap dinner", posts, k=3)

[{'post_id': 2,
  'user_id': 2,
  'topic': 'food',
  'content': 'Looking for an affordable place to eat dinner tonight. [community note 2]'},
 {'post_id': 5,
  'user_id': 5,
  'topic': 'food',
  'content': 'Looking for an affordable place to eat dinner tonight. [community note 5]'},
 {'post_id': 8,
  'user_id': 8,
  'topic': 'food',
  'content': 'Looking for an affordable place to eat dinner tonight. [community note 8]'}]

## TODO 1 — Encode text

Load the supplied pretrained encoder and create a normalized embedding matrix for all corpus texts. Keep the model call at this boundary; your retrieval functions below must not load a model or perform I/O.

In [ ]:
# Load the pretrained encoder on CPU and encode every post once.
model = SentenceTransformer(MODEL_NAME, device="cpu")

encoding_started = perf_counter()
corpus_embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
encoding_seconds = perf_counter() - encoding_started
corpus_embeddings = np.asarray(corpus_embeddings, dtype=np.float32)

assert corpus_embeddings.ndim == 2
assert corpus_embeddings.shape[0] == len(posts)
assert np.issubdtype(corpus_embeddings.dtype, np.floating)

embedding_memory_mb = corpus_embeddings.nbytes / (1024 ** 2)
print(f"Model: {MODEL_NAME} (CPU)")
print(f"Encoded {len(texts)} posts in {encoding_seconds:.2f} seconds")
print(f"Embedding matrix: {corpus_embeddings.shape}, {corpus_embeddings.dtype}")
print(f"Embedding matrix memory: {embedding_memory_mb:.2f} MiB")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Model: all-MiniLM-L6-v2 (CPU)
Encoded 1200 posts in 20.89 seconds
Embedding matrix: (1200, 384), float32
Embedding matrix memory: 1.76 MiB


## TODO 2 — Pure retrieval functions

Implement these functions with NumPy. A zero vector must not produce `NaN` or infinity. For equal scores, the lower `post_id` comes first.

In [ ]:
def cosine_similarity(query_vector, corpus_vectors):
    """Return one cosine score per corpus row; zero vectors receive score 0."""
    query = np.asarray(query_vector, dtype=np.float64).reshape(-1)
    corpus = np.asarray(corpus_vectors, dtype=np.float64)

    if corpus.ndim != 2:
        raise ValueError("corpus_vectors must be a 2D matrix.")
    if corpus.shape[1] != query.size:
        raise ValueError("Query and corpus vectors must have the same dimension.")

    query_norm = np.linalg.norm(query)
    corpus_norms = np.linalg.norm(corpus, axis=1)
    denominators = corpus_norms * query_norm

    scores = np.zeros(corpus.shape[0], dtype=np.float64)
    valid = denominators > 0
    if np.any(valid):
        scores[valid] = (corpus[valid] @ query) / denominators[valid]

    return np.clip(scores, -1.0, 1.0)


def stable_top_k(scores, post_ids, k):
    """Return indices ordered by score descending, then post_id ascending."""
    scores = np.asarray(scores)
    post_ids = np.asarray(post_ids)

    if scores.ndim != 1 or post_ids.ndim != 1:
        raise ValueError("scores and post_ids must be one-dimensional.")
    if len(scores) != len(post_ids):
        raise ValueError("scores and post_ids must have the same length.")
    if not isinstance(k, (int, np.integer)) or isinstance(k, (bool, np.bool_)):
        raise ValueError("k must be an integer.")
    if k < 0:
        raise ValueError("k must be non-negative.")
    if not np.all(np.isfinite(scores)):
        raise ValueError("scores must all be finite.")

    # lexsort uses the last key as the primary key:
    # primary = descending score; tie-break = ascending post_id.
    order = np.lexsort((post_ids, -scores))
    return order[:min(k, len(order))]


def semantic_search(query_embedding, posts, corpus_embeddings, k=5):
    """Return up to k post dictionaries with numeric cosine similarity scores."""
    if k < 0:
        raise ValueError("k must be non-negative.")
    if len(posts) != len(corpus_embeddings):
        raise ValueError("There must be one embedding row per post.")

    scores = cosine_similarity(query_embedding, corpus_embeddings)
    post_ids = np.asarray([post["post_id"] for post in posts])
    indices = stable_top_k(scores, post_ids, k)

    results = []
    for index in indices:
        result = dict(posts[int(index)])
        result["score"] = float(scores[int(index)])
        results.append(result)
    return results


## Acceptance checks

Run this cell only after implementing the TODOs. These checks are deliberately small; they verify the retrieval contract, not whether a model is magically perfect.

In [ ]:
tiny_query = np.array([1.0, 0.0])
tiny_corpus = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.0], [0.0, 0.0]])
tiny_scores = cosine_similarity(tiny_query, tiny_corpus)
assert np.allclose(tiny_scores, [1.0, 0.0, 1.0, 0.0])
assert np.isfinite(cosine_similarity(np.array([0.0, 0.0]), tiny_corpus)).all()

tie_indices = stable_top_k(np.array([0.7, 0.9, 0.9]), np.array([7, 5, 2]), k=2)
assert list(tie_indices) == [2, 1], "ties must use ascending post_id"

assert corpus_embeddings.shape[0] == len(posts)
assert np.issubdtype(corpus_embeddings.dtype, np.floating)

query_embedding = model.encode(["cheap dinner near the station"], normalize_embeddings=True)[0]
results = semantic_search(query_embedding, posts, corpus_embeddings, k=5)
assert 1 <= len(results) <= 5
assert all({"post_id", "user_id", "content", "score"} <= set(result) for result in results)
assert all(results[i]["score"] >= results[i + 1]["score"] for i in range(len(results) - 1))
assert all(np.isfinite(result["score"]) for result in results)
print("Acceptance checks passed.")

Acceptance checks passed.


## Compare search behavior

Use at least three queries. For each, compare the keyword baseline and your semantic search. Record one useful result and one surprising or weak result.

In [ ]:
comparison_started = perf_counter()

for query in queries:
    print("\\n" + "=" * 80)
    print(f"QUERY: {query!r}")

    keyword_results = keyword_search(query, posts, k=5)
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0]
    semantic_results = semantic_search(
        query_embedding, posts, corpus_embeddings, k=5
    )

    print("\\nKeyword baseline:")
    if keyword_results:
        for result in keyword_results:
            print(f"  post_id={result['post_id']} | {result['content']}")
    else:
        print("  No keyword matches.")

    print("\\nSemantic search:")
    for result in semantic_results:
        print(
            f"  score={result['score']:.4f} | post_id={result['post_id']} "
            f"| {result['content']}"
        )

comparison_seconds = perf_counter() - comparison_started
print(f"\\nComparison completed in {comparison_seconds:.2f} seconds.")
print(f"Corpus encoding time: {encoding_seconds:.2f} seconds.")
print(f"Embedding matrix memory: {embedding_memory_mb:.2f} MiB.")


\n================================================================================
QUERY: 'cheap dinner'
\nKeyword baseline:
  post_id=2 | Looking for an affordable place to eat dinner tonight. [community note 2]
  post_id=5 | Looking for an affordable place to eat dinner tonight. [community note 5]
  post_id=8 | Looking for an affordable place to eat dinner tonight. [community note 8]
  post_id=11 | Looking for an affordable place to eat dinner tonight. [community note 11]
  post_id=14 | Looking for an affordable place to eat dinner tonight. [community note 14]
\nSemantic search:
  score=0.6392 | post_id=194 | Looking for an affordable place to eat dinner tonight. [community note 194]
  score=0.6358 | post_id=197 | Looking for an affordable place to eat dinner tonight. [community note 197]
  score=0.6346 | post_id=26 | Looking for an affordable place to eat dinner tonight. [community note 26]
  score=0.6319 | post_id=2 | Looking for an affordable place to eat dinner tonight. [communit

## RESULTS

```text
RESULTS

baseline:
Keyword search finds posts that share exact words with the query.
It works for direct matches, but it may miss
semantically relevant posts when the wording differs.
In the observed "beginner running route" query, both
approaches returned posts about easy jogging routes
for beginners. For "learn Python types", both
approaches retrieved posts about Python type hints,
although the semantic search ranked these posts using
embedding similarity.

solution:
Used the pretrained SentenceTransformer model
all-MiniLM-L6-v2 to encode the 1,200-post corpus
into 384-dimensional embeddings.
The embeddings were normalized and stored as a float32
NumPy matrix. Retrieval uses cosine similarity
implemented in NumPy and deterministic top-k ordering
by descending score, with ascending post_id as the tie-breaker.

runtime:
Corpus encoding took 20.89 seconds on CPU. The comparison
of the three queries took 0.10 seconds. The first run also
downloaded the pretrained model,
so model download time is not included in the
corpus encoding measurement.

memory / model size:
The corpus embedding matrix has shape (1200, 384),
 uses float32 values, and occupies 1.76 MiB. This is
 the embedding matrix memory, not the size of the downloaded model files.

3 failure cases:
1. Repeated or near-duplicate posts can occupy
multiple top-k positions, reducing result diversity.
2. Semantically related results can receive very
similar scores, so small score differences should
not be treated as definitive evidence of relevance.
3. A query can be ambiguous or require context that
 is not present in a short post; the encoder may
 rank a plausible but not fully relevant result highly.

what I would try next:
Evaluate retrieval with a larger set of varied
queries and manually judged relevance labels.
Measure precision at k, investigate ambiguous queries,
 and consider deduplicating repeated content or
  diversifying results while preserving deterministic ranking.
```
